In [1]:
from vanna import Agent
from vanna.core.registry import ToolRegistry
from vanna.core.user import UserResolver, User, RequestContext
from vanna.tools import RunSqlTool, VisualizeDataTool
from vanna.tools.agent_memory import SaveQuestionToolArgsTool, SearchSavedCorrectToolUsesTool, SaveTextMemoryTool
from vanna.servers.fastapi import VannaFastAPIServer
from vanna.integrations.openai import OpenAILlmService
from vanna.integrations.sqlite import SqliteRunner
from vanna.integrations.local.agent_memory import DemoAgentMemory

llm = OpenAILlmService(
    model="zhipuai/glm-5",
    api_key="sk-vH7IxF0dcLUAy0HunorBGVpcLT7hY5esJQX2MQWYuiJbno66",
    base_url="http://model.mify.ai.srv/v1/"
)

In [ ]:
db_tool = RunSqlTool(
  sql_runner=SqliteRunner(
    database_path="../db/employees.db"
  ),
)

agent_memory = DemoAgentMemory(max_items=1000)
  
class SimpleUserResolver(UserResolver):
    async def resolve_user(self, request_context: RequestContext) -> User:
        user_email = request_context.get_cookie('vanna_email') or 'guest@example.com'
        group = 'admin' if user_email == 'admin@example.com' else 'user'
        return User(id=user_email, email=user_email, group_memberships=[group])
    
    
tools = ToolRegistry()
tools.register_local_tool(db_tool, access_groups=['admin', 'user'])
tools.register_local_tool(SaveQuestionToolArgsTool(), access_groups=['admin'])
tools.register_local_tool(SearchSavedCorrectToolUsesTool(), access_groups=['admin', 'user'])
tools.register_local_tool(SaveTextMemoryTool(), access_groups=['admin', 'user'])
tools.register_local_tool(VisualizeDataTool(), access_groups=['admin', 'user'])

agent = Agent(
    llm_service=llm,
    tool_registry=tools,
    user_resolver=SimpleUserResolver(),
    agent_memory=agent_memory
)

server = VannaFastAPIServer(agent)
server.run() 

INFO:     Started server process [4157308]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Your app is running at:
http://localhost:8000
INFO:     127.0.0.1:45976 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:45976 - "GET / HTTP/1.1" 200 OK
